In [ ]:
!pip install llama-cpp-python huggingface_hub

# llama.cpp CPU-Only Chat Completion Test
This notebook tests chat completion using llama.cpp running on CPU only.

In [6]:
# Verify llama-cpp-python installation
import llama_cpp
print(f'llama-cpp-python version: {llama_cpp.__version__}')

llama-cpp-python version: 0.3.25


## Download Model
Downloads a small Llama 3.2 3B Instruct GGUF model (Q4_K_M quantization ~2GB) into the `models/` folder next to this notebook.

In [7]:
# Download Llama 3.2 3B Instruct GGUF model
import os
from huggingface_hub import hf_hub_download

# Models folder next to this notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
MODELS_DIR = os.path.join(NOTEBOOK_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Model details - Llama-3.2-3B-Instruct Q4_K_M (~2GB, good quality/speed tradeoff)
REPO_ID = 'bartowski/Llama-3.2-3B-Instruct-GGUF'
FILENAME = 'Llama-3.2-3B-Instruct-Q4_K_M.gguf'
MODEL_PATH = os.path.join(MODELS_DIR, FILENAME)

if os.path.exists(MODEL_PATH):
    size_gb = os.path.getsize(MODEL_PATH) / (1024**3)
    print(f'✅ Model already exists: {MODEL_PATH}')
    print(f'   Size: {size_gb:.2f} GB')
else:
    print(f'📥 Downloading {FILENAME} from {REPO_ID}...')
    print(f'   Destination: {MODELS_DIR}')
    print(f'   This may take a few minutes (~2GB)...')
    
    downloaded_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME,
        local_dir=MODELS_DIR,
        local_dir_use_symlinks=False
    )
    
    size_gb = os.path.getsize(MODEL_PATH) / (1024**3)
    print(f'\n✅ Model downloaded successfully!')
    print(f'   Path: {MODEL_PATH}')
    print(f'   Size: {size_gb:.2f} GB')

print(f'\nModels directory contents:')
for f in os.listdir(MODELS_DIR):
    fpath = os.path.join(MODELS_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024**2)
    print(f'  {f} ({size_mb:.0f} MB)')

/home/codx-junior-projects/codx-junior/notebooks/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/codx-junior-projects/codx-junior/notebooks/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


📥 Downloading Llama-3.2-3B-Instruct-Q4_K_M.gguf from bartowski/Llama-3.2-3B-Instruct-GGUF...
   Destination: /home/codx-junior-projects/codx-junior/notebooks/llama.cpp/models
   This may take a few minutes (~2GB)...

✅ Model downloaded successfully!
   Path: /home/codx-junior-projects/codx-junior/notebooks/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf
   Size: 1.88 GB

Models directory contents:
  .cache (0 MB)
  Llama-3.2-3B-Instruct-Q4_K_M.gguf (1926 MB)


In [8]:
# Configuration
import os

# Use the downloaded model path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
MODEL_PATH = os.path.join(NOTEBOOK_DIR, 'models', 'Llama-3.2-3B-Instruct-Q4_K_M.gguf')

# CPU-only settings
N_GPU_LAYERS = 0        # 0 = CPU only, no GPU offloading
N_CTX = 2048            # Context window size
N_THREADS = os.cpu_count()  # Use all available CPU threads
N_BATCH = 512           # Batch size for prompt processing

print(f'Model path: {MODEL_PATH}')
print(f'GPU layers: {N_GPU_LAYERS} (CPU only)')
print(f'Context size: {N_CTX}')
print(f'CPU threads: {N_THREADS}')
print(f'Batch size: {N_BATCH}')

Model path: /home/codx-junior-projects/codx-junior/notebooks/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf
GPU layers: 0 (CPU only)
Context size: 2048
CPU threads: 64
Batch size: 512


In [9]:
# Check if model file exists
import os

if not os.path.exists(MODEL_PATH):
    print(f'⚠️  Model not found at: {MODEL_PATH}')
    print('\nAvailable GGUF files in common locations:')
    search_dirs = ['/models', '/home', os.path.expanduser('~'), '/tmp']
    for search_dir in search_dirs:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                for file in files:
                    if file.endswith('.gguf'):
                        print(f'  Found: {os.path.join(root, file)}')
                # Limit depth
                if root.count(os.sep) - search_dir.count(os.sep) >= 3:
                    break
else:
    size_gb = os.path.getsize(MODEL_PATH) / (1024**3)
    print(f'✅ Model found: {MODEL_PATH}')
    print(f'   Size: {size_gb:.2f} GB')

✅ Model found: /home/codx-junior-projects/codx-junior/notebooks/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf
   Size: 1.88 GB


In [10]:
# Load the model (CPU only)
from llama_cpp import Llama
import time

print('Loading model on CPU...')
start_time = time.time()

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=N_GPU_LAYERS,   # CPU only
    n_ctx=N_CTX,
    n_threads=N_THREADS,
    n_batch=N_BATCH,
    verbose=True
)

load_time = time.time() - start_time
print(f'\n✅ Model loaded successfully in {load_time:.2f}s')

llama_model_loader: loaded meta data with 35 key-value pairs and 255 tensors from /home/codx-junior-projects/codx-junior/notebooks/llama.cpp/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv   6:                            general.license str         

Loading model on CPU...


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 128254 '<|reserved_special_token_246|>' is not marked as EOG
load: control token: 128249 '<|reserved_special_token_241|>' is not marked as EOG
load: control token: 128246 '<|reserved_special_token_238|>' is not marked as EOG
load: control token: 128243 '<|reserved_special_token_235|>' is not marked as EOG
load: control token: 128242 '<|reserved_special_token_234|>' is not marked as EOG
load: control token: 128241 '<|reserved_special_token_233|>' is not marked as EOG
load: control token: 128240 '<|reserved_special_token_232|>' is not marked as EOG
load: control token: 128235 '<|reserved_special_token_227|>' is not marked as EOG
load: control token: 128231 '<|reserved_special_token_223|>' is not marked as EOG
load: control token: 128230 '<|reserved_special_token_222|>' is not marked as EOG
load: control token: 128228 '<|reserved_special_token_220|>' is not marked as EOG
load: control token: 12822


✅ Model loaded successfully in 1.84s


Using gguf chat template: {{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date

In [11]:
# Helper function for chat completion
import time

def chat_completion(messages, max_tokens=512, temperature=0.7, top_p=0.95):
    """
    Run chat completion using llama.cpp on CPU.
    
    Args:
        messages: List of message dicts with 'role' and 'content'
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        top_p: Top-p sampling
    
    Returns:
        dict with response and timing info
    """
    print(f'Running chat completion (CPU only)...')
    print(f'Messages: {len(messages)} | Max tokens: {max_tokens}')
    print('-' * 60)
    
    start_time = time.time()
    
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        stream=False
    )
    
    elapsed = time.time() - start_time
    
    content = response['choices'][0]['message']['content']
    usage = response.get('usage', {})
    prompt_tokens = usage.get('prompt_tokens', 0)
    completion_tokens = usage.get('completion_tokens', 0)
    tokens_per_sec = completion_tokens / elapsed if elapsed > 0 else 0
    
    print(f'Response:\n{content}')
    print('-' * 60)
    print(f'⏱️  Time: {elapsed:.2f}s')
    print(f'📊 Prompt tokens: {prompt_tokens} | Completion tokens: {completion_tokens}')
    print(f'🚀 Speed: {tokens_per_sec:.2f} tokens/sec (CPU)')
    
    return {
        'content': content,
        'elapsed': elapsed,
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'tokens_per_sec': tokens_per_sec,
        'raw': response
    }

print('✅ chat_completion() helper defined')

✅ chat_completion() helper defined


In [12]:
# Test 1: Basic greeting
print('=' * 60)
print('TEST 1: Basic Greeting')
print('=' * 60)

messages_1 = [
    {
        'role': 'system',
        'content': 'You are a helpful assistant. Be concise and clear.'
    },
    {
        'role': 'user',
        'content': 'Hello! Can you introduce yourself briefly?'
    }
]

result_1 = chat_completion(messages_1, max_tokens=150)

TEST 1: Basic Greeting
Running chat completion (CPU only)...
Messages: 2 | Max tokens: 150
------------------------------------------------------------


llama_perf_context_print:        load time =   20738.18 ms
llama_perf_context_print: prompt eval time =   20737.25 ms /    54 tokens (  384.02 ms per token,     2.60 tokens per second)
llama_perf_context_print:        eval time =  214728.62 ms /    53 runs   ( 4051.48 ms per token,     0.25 tokens per second)
llama_perf_context_print:       total time =  235490.13 ms /   107 tokens
llama_perf_context_print:    graphs reused =         52


Response:
I'm an artificial intelligence assistant, here to provide information and help with your questions to the best of my abilities. I don't have a personal identity, but I'm designed to be friendly and assist you in a helpful way. How can I assist you today?
------------------------------------------------------------
⏱️  Time: 235.49s
📊 Prompt tokens: 54 | Completion tokens: 53
🚀 Speed: 0.23 tokens/sec (CPU)


In [13]:
# Test 2: Technical question
print('=' * 60)
print('TEST 2: Technical Question')
print('=' * 60)

messages_2 = [
    {
        'role': 'system',
        'content': 'You are an expert software engineer. Provide clear, accurate technical answers.'
    },
    {
        'role': 'user',
        'content': 'What is the difference between a process and a thread? Give a brief explanation with a simple analogy.'
    }
]

result_2 = chat_completion(messages_2, max_tokens=300)

TEST 2: Technical Question
Running chat completion (CPU only)...
Messages: 2 | Max tokens: 300
------------------------------------------------------------


Llama.generate: 27 prefix-match hit, remaining 42 prompt tokens to eval
llama_perf_context_print:        load time =   20738.18 ms
llama_perf_context_print: prompt eval time =   21302.77 ms /    42 tokens (  507.21 ms per token,     1.97 tokens per second)
llama_perf_context_print:        eval time = 1179302.12 ms /   299 runs   ( 3944.15 ms per token,     0.25 tokens per second)
llama_perf_context_print:       total time = 1200826.15 ms /   341 tokens
llama_perf_context_print:    graphs reused =        297


Response:
**Processes vs Threads: A Simple Analogy**

Imagine you're at a coffee shop with two baristas. One barista is handling a single coffee order, and the other barista is also handling a single coffee order. In this case, each barista is a separate entity, working independently on a single task.

Now, imagine the coffee shop has a manager who divides the workload between the two baristas. The manager might assign multiple coffee orders to each barista, allowing them to work on multiple orders simultaneously. In this scenario, each barista is still working on separate coffee orders, but they're now handling multiple tasks.

**Processes**

In the context of operating systems, a process is like a single barista (or the manager) handling a single coffee order. A process is a separate entity, with its own memory space, that runs on a CPU. Each process has its own program counter, memory, and resources.

**Threads**

In contrast, a thread is like a multitasking assistant (or a helper b

In [ ]:
# Test 3: Multi-turn conversation
print('=' * 60)
print('TEST 3: Multi-turn Conversation')
print('=' * 60)

messages_3 = [
    {
        'role': 'system',
        'content': 'You are a helpful coding assistant specialized in Python.'
    },
    {
        'role': 'user',
        'content': 'Write a simple Python function to calculate fibonacci numbers.'
    },
    {
        'role': 'assistant',
        'content': 'Here is a simple Python function:\n```python\ndef fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)\n```'
    },
    {
        'role': 'user',
        'content': 'Now optimize it using memoization.'
    }
]

result_3 = chat_completion(messages_3, max_tokens=400)

In [ ]:
# Test 4: Streaming chat completion
print('=' * 60)
print('TEST 4: Streaming Chat Completion')
print('=' * 60)

messages_4 = [
    {
        'role': 'system',
        'content': 'You are a creative writer.'
    },
    {
        'role': 'user',
        'content': 'Write a 3-sentence story about an AI running on a CPU.'
    }
]

print('Streaming response:')
print('-' * 60)

start_time = time.time()
full_response = ''
token_count = 0

stream = llm.create_chat_completion(
    messages=messages_4,
    max_tokens=200,
    temperature=0.8,
    stream=True
)

for chunk in stream:
    delta = chunk['choices'][0].get('delta', {})
    content = delta.get('content', '')
    if content:
        print(content, end='', flush=True)
        full_response += content
        token_count += 1

elapsed = time.time() - start_time
print(f'\n{"-" * 60}')
print(f'⏱️  Streaming time: {elapsed:.2f}s')
print(f'🚀 Approx speed: {token_count/elapsed:.2f} tokens/sec (CPU)')

In [ ]:
# Summary Report
print('=' * 60)
print('SUMMARY: CPU-Only llama.cpp Performance')
print('=' * 60)

results = [
    ('Basic Greeting', result_1),
    ('Technical Question', result_2),
    ('Multi-turn Conversation', result_3),
]

total_time = 0
total_tokens = 0

print(f'{"Test":<30} {"Time(s)":<12} {"Tokens":<10} {"Tok/s":<10}')
print('-' * 62)

for name, result in results:
    print(f'{name:<30} {result["elapsed"]:<12.2f} {result["completion_tokens"]:<10} {result["tokens_per_sec"]:<10.2f}')
    total_time += result['elapsed']
    total_tokens += result['completion_tokens']

print('-' * 62)
avg_speed = total_tokens / total_time if total_time > 0 else 0
print(f'{"TOTAL/AVG":<30} {total_time:<12.2f} {total_tokens:<10} {avg_speed:<10.2f}')
print(f'\n🖥️  Backend: CPU only (n_gpu_layers=0)')
print(f'🧵 Threads used: {N_THREADS}')
print(f'📐 Context size: {N_CTX}')
print(f'📦 Model: {MODEL_PATH}')